In [ ]:
!pip install torch transformers gradio pillow -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.9/321.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
pip install easyocr -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.9/422.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.6/969.6 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.6/286.6 kB 24.4 MB/s eta 0:00:00


In [ ]:
pip install -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.1/456.1 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.59.9
    Uninstalling openai-1.59.9:
      Successfully uninstalled openai-1.59.9


### Returns answer with explanation

In [ ]:
from google.colab import userdata
API_KEY = userdata.get('Secret_Key') # replace secret key with yours

In [ ]:
import gradio as gr
from PIL import Image
import easyocr
from openai import OpenAI

# Initialize EasyOCR reader (English)
reader = easyocr.Reader(['en'])

# DeepSeek API client configuration
client = OpenAI(api_key=DeepSeek_API_KEY, base_url="https://api.deepseek.com")

def solve_puzzle(image):
    """Extracts the puzzle from the image and sends it to DeepSeek for solving."""
    try:
        # 1. Save the uploaded image temporarily; EasyOCR uses file paths
        image_path = "uploaded_image.png"
        image.save(image_path)

        # 2. Extract text from the image using EasyOCR
        results = reader.readtext(image_path)
        extracted_text = " ".join([res[1] for res in results])

        # Standardize the text to avoid misinterpretation of "??" as "?"
        extracted_text = extracted_text.replace('??', '?')

        if "?" not in extracted_text:
            extracted_text += "?"

        print("Extracted Text:", extracted_text)  # Debugging output

        # 3. Refine the extracted text to standardize expressions
        refined_text = extracted_text.replace('x', '*').replace('X', '*').replace('=', ' = ').strip()
        print("Refined Text:", refined_text)  # Debugging output

        # 4. Compose the user message with detailed reasoning instructions
        puzzle_prompt = (
            f"You are an AI specialized in solving simple and complex maths puzzles. Analyze the following, identify hidden patterns or rules, and provide the missing value with step-by-step reasoning in text format. Do not return the answer in LaTeX."
            f"\nPuzzle:\n{refined_text}\n"
            "Format your response strictly as follows:\n"
            "1. **Given sequence**:\n   - (original sequences)\n"
            "2. **Pattern Identified**:\n   (explain the hidden logic)\n"
            "3. **Step-by-step Calculation**:\n   - For (input values):\n     (calculation and result)\n"
            "4. **Final Answer**:\n     (Answer = X)"
        )

        # 5. Initial message to DeepSeek API
        messages = [{"role": "user", "content": puzzle_prompt}]

        # 6. Send the request to DeepSeek API
        response = client.chat.completions.create(
            model="deepseek-reasoner",
            messages=messages,
            temperature=0,  # Reduce randomness for direct answers
            max_tokens=500  # Allow enough tokens for explanation
        )

        # 7. Process the response from DeepSeek API
        reasoning_content = response.choices[0].message.content.strip()

        # Return the formatted response
        return reasoning_content if reasoning_content else "Error: No response content."

    except Exception as e:
        return f"Error: {str(e)}"

# Gradio interface
interface = gr.Interface(
    fn=solve_puzzle,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Math Puzzle Solver with EasyOCR & DeepSeek-R1",
    description="Upload an image of a math puzzle, and the model will solve it with step-by-step reasoning."
)

interface.launch(debug=True)


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

KeyboardInterrupt: 

### Returns just answer

In [ ]:
import gradio as gr
from PIL import Image
import easyocr
from openai import OpenAI

# Initialize EasyOCR reader (English)
reader = easyocr.Reader(['en'])

# DeepSeek API client configuration
client = OpenAI(api_key=DeepSeek_API_KEY, base_url="https://api.deepseek.com")

def solve_puzzle(image):
    """Extracts the puzzle from the image and sends it to DeepSeek for solving."""
    try:
        # 1. Save the uploaded image temporarily; EasyOCR uses file paths
        image_path = "uploaded_image.png"
        image.save(image_path)

        # 2. Extract text from the image using EasyOCR
        results = reader.readtext(image_path)
        extracted_text = " ".join([res[1] for res in results])

        # Standardize the text to avoid misinterpretation of "??" as "?"
        extracted_text = extracted_text.replace('??', '?')

        if "?" not in extracted_text:
            extracted_text += "?"

        print("Extracted Text:", extracted_text)  # Debugging output

        # 3. Refine the extracted text to standardize expressions
        refined_text = extracted_text.replace('x', '*').replace('X', '*').replace('=', ' = ').strip()
        print("Refined Text:", refined_text)  # Debugging output

        # 4. Compose the user message with detailed reasoning instructions
        puzzle_prompt = (
            f"You are an AI specialized in solving simple and complex math puzzles. Analyze the following puzzle, identify hidden patterns or rules, and provide only the final missing value. Do not include step-by-step reasoning or explanations."
            f"\nPuzzle:\n{refined_text}\n"
            "Format your response strictly as follows:\n"
            "Final Answer:"
        )

        # 5. Initial message to DeepSeek API
        messages = [{"role": "user", "content": puzzle_prompt}]

        # 6. Send the request to DeepSeek API
        response = client.chat.completions.create(
            model="deepseek-reasoner",
            messages=messages,
            temperature=0,  # Reduce randomness for direct answers
            max_tokens=500  # Allow enough tokens for explanation
        )

        # 7. Process the response from DeepSeek API
        reasoning_content = response.choices[0].message.content.strip()

        # Return the formatted response
        return reasoning_content if reasoning_content else "Error: No response content."

    except Exception as e:
        return f"Error: {str(e)}"

# Gradio interface
interface = gr.Interface(
    fn=solve_puzzle,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Math Puzzle Solver with EasyOCR & DeepSeek-R1",
    description="Upload an image of a math puzzle, and the model will solve it with step-by-step reasoning."
)

interface.launch(debug=True)


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ffa7540fc46bf40ff6.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Extracted Text: 1 + 4 =5 2 + 5 = 12 3 + 6 = 21 8 + 11 = ?
Refined Text: 1 + 4  = 5 2 + 5  =  12 3 + 6  =  21 8 + 11  =  ?
Extracted Text: 2 5 3 6 1 8 2 3 7?
Refined Text: 2 5 3 6 1 8 2 3 7?
